# Tabular ML: XGBoost y LightGBM

En este notebook exploraremos los dos frameworks de **gradient boosting** mas populares para datos
tabulares: **XGBoost** y **LightGBM**. Estos algoritmos siguen siendo los reyes indiscutibles para
datos tabulares en la mayoria de competiciones y aplicaciones de negocio.

**Objetivos:**
- Entender la intuicion detras del gradient boosting
- Entrenar y comparar XGBoost vs LightGBM
- Aplicar feature engineering efectivo
- Optimizar hiperparametros con Optuna
- Interpretar modelos con SHAP
- Construir un pipeline completo para produccion

## Gradient Boosting: intuicion

### Como funciona el boosting

A diferencia de **bagging** (Random Forest) que entrena arboles en paralelo, el **boosting**
entrena arboles **secuencialmente**, donde cada nuevo arbol corrige los errores del anterior.

```
Prediccion final = Arbol_1 + lr * Arbol_2 + lr * Arbol_3 + ... + lr * Arbol_N
```

### El proceso paso a paso:
1. **Iniciar** con una prediccion base (ej: la media de `y`)
2. **Calcular residuos**: diferencia entre prediccion actual y valor real
3. **Entrenar un arbol** para predecir los residuos
4. **Actualizar la prediccion** sumando la prediccion del nuevo arbol (escalada por learning rate)
5. **Repetir** pasos 2-4 hasta alcanzar el numero de arboles deseado

### XGBoost vs LightGBM

| Caracteristica | XGBoost | LightGBM |
|---------------|---------|----------|
| Crecimiento de arbol | Level-wise (por nivel) | Leaf-wise (por hoja) |
| Velocidad | Rapido | Mas rapido (2-10x) |
| Memoria | Mas memoria | Menos memoria |
| Categoricas | Requiere encoding | Soporte nativo |
| Overfitting | Mas controlado | Puede sobreajustar mas en datos pequenos |
| GPU support | Si | Si |
| Missing values | Manejo nativo | Manejo nativo |

In [ ]:
# !pip install xgboost lightgbm optuna shap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    roc_auc_score, roc_curve
)

# Create a synthetic business dataset: Customer Churn Prediction
np.random.seed(42)
n_samples = 5000

data = {
    "tenure_months": np.random.exponential(24, n_samples).clip(1, 72).astype(int),
    "monthly_charges": np.random.normal(65, 25, n_samples).clip(20, 150),
    "total_charges": None,  # Will derive from tenure and monthly
    "contract_type": np.random.choice(["month-to-month", "one-year", "two-year"], n_samples, p=[0.5, 0.3, 0.2]),
    "payment_method": np.random.choice(["credit_card", "bank_transfer", "electronic_check", "mailed_check"], n_samples),
    "num_support_tickets": np.random.poisson(2, n_samples),
    "num_products": np.random.randint(1, 6, n_samples),
    "has_partner": np.random.choice([0, 1], n_samples, p=[0.4, 0.6]),
    "age": np.random.normal(45, 15, n_samples).clip(18, 80).astype(int),
    "satisfaction_score": np.random.uniform(1, 10, n_samples),
}

df = pd.DataFrame(data)
df["total_charges"] = df["tenure_months"] * df["monthly_charges"] * np.random.uniform(0.9, 1.1, n_samples)

# Create churn label based on realistic patterns
churn_prob = (
    0.3
    - 0.004 * df["tenure_months"]
    + 0.003 * df["monthly_charges"]
    + 0.05 * df["num_support_tickets"]
    - 0.03 * df["num_products"]
    - 0.02 * df["satisfaction_score"]
    + 0.15 * (df["contract_type"] == "month-to-month").astype(int)
    + 0.05 * (df["payment_method"] == "electronic_check").astype(int)
)
churn_prob = 1 / (1 + np.exp(-churn_prob))  # Sigmoid
df["churn"] = (np.random.random(n_samples) < churn_prob).astype(int)

# EDA
print("Dataset: Customer Churn Prediction")
print(f"Shape: {df.shape}")
print(f"\nChurn rate: {df['churn'].mean():.1%}")
print(f"\nInfo:")
print(df.describe().round(2))

# Visualize key features
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Churn distribution
df["churn"].value_counts().plot.bar(ax=axes[0, 0], color=["#51cf66", "#ff6b6b"])
axes[0, 0].set_title("Distribucion de Churn")
axes[0, 0].set_xticklabels(["No Churn", "Churn"], rotation=0)

# Tenure by churn
for label, group in df.groupby("churn"):
    axes[0, 1].hist(group["tenure_months"], bins=20, alpha=0.6,
                     label=f"Churn={label}")
axes[0, 1].set_title("Tenure por Churn")
axes[0, 1].legend()

# Monthly charges by churn
for label, group in df.groupby("churn"):
    axes[0, 2].hist(group["monthly_charges"], bins=20, alpha=0.6,
                     label=f"Churn={label}")
axes[0, 2].set_title("Monthly Charges por Churn")
axes[0, 2].legend()

# Contract type
pd.crosstab(df["contract_type"], df["churn"], normalize="index").plot.bar(
    ax=axes[1, 0], color=["#51cf66", "#ff6b6b"])
axes[1, 0].set_title("Churn Rate por Contract Type")
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=45)

# Support tickets
df.groupby("num_support_tickets")["churn"].mean().plot.bar(ax=axes[1, 1], color="#748ffc")
axes[1, 1].set_title("Churn Rate por Support Tickets")

# Satisfaction score
for label, group in df.groupby("churn"):
    axes[1, 2].hist(group["satisfaction_score"], bins=20, alpha=0.6,
                     label=f"Churn={label}")
axes[1, 2].set_title("Satisfaction Score por Churn")
axes[1, 2].legend()

plt.suptitle("Analisis Exploratorio - Customer Churn", fontsize=16)
plt.tight_layout()
plt.show()

## XGBoost

**XGBoost** (eXtreme Gradient Boosting) fue creado por Tianqi Chen en 2014 y rapidamente se
convirtio en el algoritmo dominante en competiciones de Kaggle para datos tabulares.

Caracteristicas clave:
- Regularizacion L1 y L2 integrada
- Manejo nativo de valores faltantes
- Paralelizacion eficiente
- Poda de arboles (tree pruning)

In [ ]:
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

# Prepare data: encode categorical variables
df_encoded = df.copy()
label_encoders = {}
for col in ["contract_type", "payment_method"]:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le

# Split features and target
X = df_encoded.drop("churn", axis=1)
y = df_encoded["churn"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.1%}")
print(f"Test churn rate:  {y_test.mean():.1%}")

# Train XGBoost classifier
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,       # L1 regularization
    reg_lambda=1.0,      # L2 regularization
    random_state=42,
    eval_metric="logloss",
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=50,
)

# Evaluate
xgb_preds = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

print(f"\n{'=' * 50}")
print("XGBoost - Resultados")
print(f"{'=' * 50}")
print(f"Accuracy: {accuracy_score(y_test, xgb_preds):.4f}")
print(f"F1-Score: {f1_score(y_test, xgb_preds):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, xgb_proba):.4f}")

# Feature importance
fig, ax = plt.subplots(figsize=(10, 6))
importance = pd.Series(
    xgb_model.feature_importances_,
    index=X.columns
).sort_values(ascending=True)

importance.plot.barh(ax=ax, color="#748ffc")
ax.set_title("XGBoost - Feature Importance", fontsize=14)
ax.set_xlabel("Importance (gain)")
plt.tight_layout()
plt.show()

## LightGBM

**LightGBM** (Light Gradient Boosting Machine) fue desarrollado por Microsoft en 2017.
Es conocido por ser significativamente mas rapido que XGBoost, especialmente en datasets grandes.

Innovaciones clave:
- **Leaf-wise growth**: Crece la hoja que mas reduce el error (vs level-wise en XGBoost)
- **GOSS**: Gradient-based One-Side Sampling (se enfoca en gradientes grandes)
- **EFB**: Exclusive Feature Bundling (agrupa features mutuamente excluyentes)
- **Soporte nativo de categoricas**: No necesita label encoding

In [ ]:
import lightgbm as lgb
import time

# LightGBM can handle categoricals natively
# Let's use the original data with category dtype
df_lgb = df.copy()
for col in ["contract_type", "payment_method"]:
    df_lgb[col] = df_lgb[col].astype("category")

X_lgb = df_lgb.drop("churn", axis=1)
y_lgb = df_lgb["churn"]

X_train_lgb, X_test_lgb, y_train_lgb, y_test_lgb = train_test_split(
    X_lgb, y_lgb, test_size=0.2, random_state=42, stratify=y_lgb
)

# Train LightGBM classifier
lgb_model = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1,  # Suppress warnings
)

start_lgb = time.time()
lgb_model.fit(
    X_train_lgb, y_train_lgb,
    eval_set=[(X_test_lgb, y_test_lgb)],
)
lgb_time = time.time() - start_lgb

# Evaluate
lgb_preds = lgb_model.predict(X_test_lgb)
lgb_proba = lgb_model.predict_proba(X_test_lgb)[:, 1]

print(f"\n{'=' * 50}")
print("LightGBM - Resultados")
print(f"{'=' * 50}")
print(f"Accuracy: {accuracy_score(y_test_lgb, lgb_preds):.4f}")
print(f"F1-Score: {f1_score(y_test_lgb, lgb_preds):.4f}")
print(f"ROC-AUC:  {roc_auc_score(y_test_lgb, lgb_proba):.4f}")

# Compare XGBoost vs LightGBM
print(f"\n{'=' * 60}")
print("COMPARACION: XGBoost vs LightGBM")
print(f"{'=' * 60}")
print(f"{'Metrica':<15} {'XGBoost':>10} {'LightGBM':>10}")
print(f"{'-'*35}")
print(f"{'Accuracy':<15} {accuracy_score(y_test, xgb_preds):>10.4f} {accuracy_score(y_test_lgb, lgb_preds):>10.4f}")
print(f"{'F1-Score':<15} {f1_score(y_test, xgb_preds):>10.4f} {f1_score(y_test_lgb, lgb_preds):>10.4f}")
print(f"{'ROC-AUC':<15} {roc_auc_score(y_test, xgb_proba):>10.4f} {roc_auc_score(y_test_lgb, lgb_proba):>10.4f}")

# ROC Curves comparison
fig, ax = plt.subplots(figsize=(8, 6))

fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_proba)
fpr_lgb, tpr_lgb, _ = roc_curve(y_test_lgb, lgb_proba)

ax.plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUC={roc_auc_score(y_test, xgb_proba):.3f})")
ax.plot(fpr_lgb, tpr_lgb, label=f"LightGBM (AUC={roc_auc_score(y_test_lgb, lgb_proba):.3f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.5)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Curvas ROC: XGBoost vs LightGBM")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Feature Engineering

El **feature engineering** es frecuentemente lo que marca la diferencia entre un modelo bueno y uno
excelente. Consiste en crear nuevas variables a partir de las existentes que capturen patrones
que el modelo no puede descubrir facilmente por si solo.

### Tecnicas comunes:
- **Interacciones**: Multiplicar o dividir features entre si
- **Agregaciones**: Estadisticas por grupo (media, mediana, conteo)
- **Features temporales**: Dia de la semana, mes, antiguedad
- **Binning**: Discretizar variables continuas
- **Ratios**: Proporciones entre variables

In [ ]:
# Feature Engineering
df_fe = df_encoded.copy()

# Interaction features
df_fe["charge_per_month_ratio"] = df_fe["total_charges"] / (df_fe["tenure_months"] + 1)
df_fe["products_per_tenure"] = df_fe["num_products"] / (df_fe["tenure_months"] + 1)
df_fe["tickets_per_tenure"] = df_fe["num_support_tickets"] / (df_fe["tenure_months"] + 1)

# Ratio features
df_fe["charge_to_satisfaction"] = df_fe["monthly_charges"] / (df_fe["satisfaction_score"] + 0.1)
df_fe["tickets_to_products"] = df_fe["num_support_tickets"] / (df_fe["num_products"] + 0.1)

# Binned features
df_fe["tenure_bin"] = pd.cut(df_fe["tenure_months"], bins=[0, 6, 12, 24, 48, 72], labels=False)
df_fe["age_bin"] = pd.cut(df_fe["age"], bins=[18, 30, 45, 60, 80], labels=False)

# Polynomial features (key interactions)
df_fe["tenure_x_satisfaction"] = df_fe["tenure_months"] * df_fe["satisfaction_score"]
df_fe["charges_x_tickets"] = df_fe["monthly_charges"] * df_fe["num_support_tickets"]

# Boolean flags
df_fe["is_new_customer"] = (df_fe["tenure_months"] <= 6).astype(int)
df_fe["is_high_spender"] = (df_fe["monthly_charges"] > df_fe["monthly_charges"].quantile(0.75)).astype(int)
df_fe["has_many_tickets"] = (df_fe["num_support_tickets"] > 3).astype(int)

print(f"Features originales: {X.shape[1]}")
print(f"Features con engineering: {df_fe.shape[1] - 1}")
print(f"\nNuevos features:")
new_features = [c for c in df_fe.columns if c not in df_encoded.columns]
for f in new_features:
    print(f"  - {f}")

# Split and retrain
X_fe = df_fe.drop("churn", axis=1)
y_fe = df_fe["churn"]
X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(
    X_fe, y_fe, test_size=0.2, random_state=42, stratify=y_fe
)

# Retrain XGBoost with engineered features
xgb_fe = xgb.XGBClassifier(
    n_estimators=200, max_depth=5, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    eval_metric="logloss",
)
xgb_fe.fit(X_train_fe, y_train_fe, eval_set=[(X_test_fe, y_test_fe)], verbose=False)

xgb_fe_preds = xgb_fe.predict(X_test_fe)
xgb_fe_proba = xgb_fe.predict_proba(X_test_fe)[:, 1]

# Compare
print(f"\n{'=' * 60}")
print("IMPACTO DEL FEATURE ENGINEERING")
print(f"{'=' * 60}")
print(f"{'Metrica':<15} {'Sin FE':>10} {'Con FE':>10} {'Mejora':>10}")
print(f"{'-'*45}")

acc_before = accuracy_score(y_test, xgb_preds)
acc_after = accuracy_score(y_test_fe, xgb_fe_preds)
f1_before = f1_score(y_test, xgb_preds)
f1_after = f1_score(y_test_fe, xgb_fe_preds)
auc_before = roc_auc_score(y_test, xgb_proba)
auc_after = roc_auc_score(y_test_fe, xgb_fe_proba)

print(f"{'Accuracy':<15} {acc_before:>10.4f} {acc_after:>10.4f} {acc_after-acc_before:>+10.4f}")
print(f"{'F1-Score':<15} {f1_before:>10.4f} {f1_after:>10.4f} {f1_after-f1_before:>+10.4f}")
print(f"{'ROC-AUC':<15} {auc_before:>10.4f} {auc_after:>10.4f} {auc_after-auc_before:>+10.4f}")

## Hyperparameter Tuning con Optuna

**Optuna** es un framework moderno de optimizacion de hiperparametros que usa algoritmos
bayesianos (TPE) para buscar de forma inteligente las mejores configuraciones.

### Ventajas sobre Grid Search y Random Search:
- **Mas eficiente**: Aprende de evaluaciones previas
- **Pruning**: Detiene trials poco prometedores tempranamente
- **Flexible**: Soporta espacios de busqueda condicionales
- **Visualizacion**: Herramientas integradas de analisis

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score

# Suppress Optuna info logs for cleaner output
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    """Optuna objective function for XGBoost hyperparameter optimization."""
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
    }

    model = xgb.XGBClassifier(
        **params,
        random_state=42,
        eval_metric="logloss",
    )

    # 3-fold cross-validation for speed
    scores = cross_val_score(
        model, X_train_fe, y_train_fe,
        cv=3, scoring="roc_auc", n_jobs=-1
    )
    return scores.mean()

# Run optimization
study = optuna.create_study(direction="maximize", study_name="xgboost_churn")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"\nMejor AUC encontrado: {study.best_value:.4f}")
print(f"\nMejores hiperparametros:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

# Plot optimization history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Optimization history
trials = study.trials
values = [t.value for t in trials if t.value is not None]
best_values = [max(values[:i+1]) for i in range(len(values))]

axes[0].plot(values, "bo-", alpha=0.5, markersize=4, label="Trial value")
axes[0].plot(best_values, "r-", linewidth=2, label="Best value")
axes[0].set_xlabel("Trial")
axes[0].set_ylabel("ROC-AUC")
axes[0].set_title("Historial de optimizacion")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Parameter importance
param_importance = optuna.importance.get_param_importances(study)
params_sorted = sorted(param_importance.items(), key=lambda x: x[1])
names, importances = zip(*params_sorted)
axes[1].barh(names, importances, color="#748ffc")
axes[1].set_xlabel("Importancia")
axes[1].set_title("Importancia de hiperparametros")

plt.tight_layout()
plt.show()

## Interpretabilidad con SHAP

**SHAP** (SHapley Additive exPlanations) es una herramienta basada en teoria de juegos para
explicar las predicciones de cualquier modelo de ML.

### Conceptos clave:
- **SHAP value**: Contribucion de cada feature a la prediccion de una instancia especifica
- **Valores positivos**: Empujan la prediccion hacia la clase positiva (churn)
- **Valores negativos**: Empujan la prediccion hacia la clase negativa (no churn)
- **Summary plot**: Vision global de que features son mas importantes y como afectan
- **Waterfall plot**: Explicacion detallada de una prediccion individual

In [ ]:
import shap

# Train final model with best params
best_model = xgb.XGBClassifier(
    **study.best_params,
    random_state=42,
    eval_metric="logloss",
)
best_model.fit(X_train_fe, y_train_fe, verbose=False)

# Create SHAP explainer for tree-based models
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_fe)

# Summary plot: global feature importance with direction
print("SHAP Summary Plot - Importancia global de features")
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_fe, show=False, max_display=15)
plt.title("SHAP Summary Plot")
plt.tight_layout()
plt.show()

# Waterfall plot: explain a single prediction
print("\nSHAP Waterfall - Explicacion de una prediccion individual")
# Pick a churned customer
churn_idx = np.where(y_test_fe.values == 1)[0][0]
print(f"  Cliente index: {churn_idx}")
print(f"  Prediccion: {'CHURN' if best_model.predict(X_test_fe.iloc[[churn_idx]])[0] else 'NO CHURN'}")
print(f"  Probabilidad churn: {best_model.predict_proba(X_test_fe.iloc[[churn_idx]])[0][1]:.3f}")

plt.figure(figsize=(10, 6))
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[churn_idx],
        base_values=explainer.expected_value,
        data=X_test_fe.iloc[churn_idx],
        feature_names=X_test_fe.columns.tolist()
    ),
    show=False,
    max_display=12
)
plt.title("Waterfall Plot - Por que este cliente tiene churn?")
plt.tight_layout()
plt.show()

# Dependence plot: relationship between a feature and SHAP values
print("\nSHAP Dependence Plot - Relacion feature-impacto")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

shap.dependence_plot(
    "tenure_months", shap_values, X_test_fe,
    interaction_index="monthly_charges",
    ax=axes[0], show=False
)
axes[0].set_title("SHAP: Tenure vs Impacto")

shap.dependence_plot(
    "satisfaction_score", shap_values, X_test_fe,
    interaction_index="num_support_tickets",
    ax=axes[1], show=False
)
axes[1].set_title("SHAP: Satisfaction vs Impacto")

plt.tight_layout()
plt.show()

## Pipeline completo para produccion

Para llevar un modelo a produccion, necesitamos encapsular todo el procesamiento en un
**pipeline** reproducible que incluya:
- Preprocesamiento (scaling, encoding, feature engineering)
- Modelo entrenado
- Serializacion para despliegue

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.model_selection import cross_val_score
import joblib

# Define column groups
numeric_features = [
    "tenure_months", "monthly_charges", "total_charges",
    "num_support_tickets", "num_products", "age", "satisfaction_score"
]
categorical_features = ["contract_type", "payment_method"]
binary_features = ["has_partner"]

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_features),
        ("bin", "passthrough", binary_features),
    ]
)

# Full pipeline: preprocessing + model
full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", xgb.XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        random_state=42,
        eval_metric="logloss",
    )),
])

# Use original DataFrame (before any encoding)
X_original = df.drop("churn", axis=1)
y_original = df["churn"]

# Cross-validation with the full pipeline
cv_scores = cross_val_score(
    full_pipeline, X_original, y_original,
    cv=5, scoring="roc_auc", n_jobs=-1
)

print("Cross-Validation con Pipeline completo:")
print(f"  ROC-AUC: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print(f"  Scores por fold: {[f'{s:.4f}' for s in cv_scores]}")

# Train final pipeline on all data
full_pipeline.fit(X_original, y_original)

# Save pipeline with joblib
model_path = "churn_pipeline.joblib"
joblib.dump(full_pipeline, model_path)
print(f"\nPipeline guardado en: {model_path}")

# Verify: load and predict
loaded_pipeline = joblib.load(model_path)

# Predict on a new customer (raw data, no preprocessing needed)
new_customer = pd.DataFrame([{
    "tenure_months": 3,
    "monthly_charges": 89.99,
    "total_charges": 270.0,
    "contract_type": "month-to-month",
    "payment_method": "electronic_check",
    "num_support_tickets": 5,
    "num_products": 1,
    "has_partner": 0,
    "age": 28,
    "satisfaction_score": 3.5,
}])

prediction = loaded_pipeline.predict(new_customer)[0]
probability = loaded_pipeline.predict_proba(new_customer)[0][1]

print(f"\nPrediccion para nuevo cliente:")
print(f"  Churn: {'SI' if prediction else 'NO'}")
print(f"  Probabilidad: {probability:.1%}")

## Resumen: checklist para proyecto tabular

### 1. Exploracion y preparacion
- [ ] Entender el problema de negocio y la metrica objetivo
- [ ] EDA: distribuciones, correlaciones, valores faltantes
- [ ] Definir estrategia de validacion (temporal, stratified, group)
- [ ] Tratar valores faltantes y outliers

### 2. Feature engineering
- [ ] Crear features de interaccion y ratios
- [ ] Features temporales si aplica
- [ ] Encoding de categoricas (ordinal, target, frequency)
- [ ] Feature selection: eliminar features irrelevantes o redundantes

### 3. Modelado
- [ ] Baseline con modelo simple (logistic regression)
- [ ] Entrenar XGBoost y LightGBM
- [ ] Optimizar hiperparametros con Optuna (50-100 trials)
- [ ] Ensemble si la mejora justifica la complejidad

### 4. Evaluacion
- [ ] Cross-validation robusta (5-10 folds)
- [ ] Metricas relevantes al negocio (AUC, F1, precision@k)
- [ ] Analisis de errores: donde falla el modelo?
- [ ] SHAP: interpretabilidad y validacion de logica

### 5. Produccion
- [ ] Pipeline sklearn (ColumnTransformer + modelo)
- [ ] Serializar con joblib/pickle
- [ ] Tests unitarios del pipeline
- [ ] Monitoreo de drift de datos y modelo
- [ ] Plan de reentrenamiento periodico